#### NOTEBOOK CELL TO PYTHON SCRIPT 

#### GOING MODULAR

##### 1. GET DATA

In [1]:
# downloading image dataset 
import os 
import requests 
import zipfile
from pathlib import Path 

data_path = Path("data/")
image_path = data_path/"pizza_steak_sushi"

if data_path.is_dir():
    print(f"{image_path} aleardy exists")
    
else: 
    print(f"Didn't find {image_path}. Downloading...")
    image_path.mkdir(parents=True, 
                     exist_ok=True)
    
    with open(data_path/"pizza_steak_sushi", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)
        
    # unzip file 
    with zipfile.ZipFile(data_path/"pizza_steak_sushi.zip", "r") as zip_ref: 
        print("Unzipping pizza, steak, sushi data...") 
        zip_ref.extractall(image_path)

data/pizza_steak_sushi aleardy exists


#### 2. CREATE DATASET AND DATALOADER

In [2]:
# import os
# os.makedirs("going_modular", exist_ok=True)

In [13]:
%%writefile going_modular/data_setup.py
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(train_dir: str,
                       test_dir: str,
                       transform: transforms.Compose,
                       batch_size: int,
                       num_workers: int = NUM_WORKERS):
    """Creates training and testing DataLoaders.

    Takes in a training directory and testing directory path and turns
    them into PyTorch Datasets and then into PyTorch DataLoaders.

    Args:
        train_dir: Path to training directory.
        test_dir: Path to testing directory.
        transform: torchvision transforms to perform on training and testing data.
        batch_size: Number of samples per batch.
        num_workers: Number of workers per DataLoader.

    Returns:
        train_dataloader, test_dataloader, class_names
    """

    # Use ImageFolder to create dataset
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    # Class names
    class_names = train_data.classes

    # Create DataLoaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_dataloader, test_dataloader, class_names


Overwriting going_modular/data_setup.py


In [15]:
# from going_modular import data_setup 
# train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(...)

#### 2. MAKING MODEL model_builder.py

In [20]:
%%writefile going_modular/model_builder.py 

import torch 
from torch import nn

class TinyVGG(nn.Module):
    """Creates the TinyVGG architecture.
    
    Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
    See the original architecture here: https://poloclub.github.io/cnn-explainer/
    
    Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
    """

    def __init__(self, input_shape:int, 
                hidden_units:int, 
                output_shape:int) -> None:
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, 
                     out_channels=hidden_units, 
                     kernel_size=3, 
                     stride=1, 
                     padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, 
                     out_channels=hidden_units, 
                     kernel_size=3, 
                     stride=1, 
                     padding=0),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, 
                     out_channels=hidden_units, 
                     kernel_size=3, 
                     stride=1, 
                     padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, 
                     out_channels=hidden_units, 
                     kernel_size=3, 
                     stride=1, 
                     padding=0),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(), 
            nn.Linear(in_features=hidden_units *13*13, 
                     out_features=output_shape)
        )

    def forward(self, x:torch.Tensor):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)

        return x
    

Overwriting going_modular/model_builder.py


In [21]:
import torch 
from going_modular import model_builder

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

model = model_builder.TinyVGG(input_shape=3, 
                             hidden_units=10, 
                             output_shape=3).to(device)

#### 3. CREATING train_step and test_step with train() function to combine them

In [25]:
%%writefile going_modular/engine.py

import torch 
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model, dataloader, loss_fn, optimizer, device) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).
    
    Args:
    model: A PyTorch model to be trained.
    dataloader: A DataLoader instance for the model to be trained on.
    loss_fn: A PyTorch loss function to minimize.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    device: A target device to compute on (e.g. "cuda" or "cpu").
    
    Returns:
    A tuple of training loss and training accuracy metrics.
    In the form (train_loss, train_accuracy). For example:

    (0.1112, 0.8743)
    """
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_preds = model(X)
        loss = loss_fn(y_preds, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_preds_class = torch.argmax(torch.softmax(y_preds, dim=1), dim=1)
        train_acc += (y_preds_class == y).sum().item()/len(y_preds)

    train_loss += train_loss/len(dataloader)
    train_acc += train_acc/len(dataloader)
    return train_acc, train_loss

def test_step(model, dataloader, loss_fn, device) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.
    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
    model: A PyTorch model to be tested.
    dataloader: A DataLoader instance for the model to be tested on.
    loss_fn: A PyTorch loss function to calculate loss on the test data.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A tuple of testing loss and testing accuracy metrics.
    In the form (test_loss, test_accuracy). For example:

    (0.0223, 0.8985)
    """
    model.eval()
    with torch.inference_mode():
        test_loss, test_acc = 0, 0
        for batch, (X,y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            y_pred_logits = model(X)
            loss = loss_fn(y_preds, y)
            test_loss += loss.item()
            test_preds_class = torch.argmax(torch.softmax(y_pred_logits, dim=1), dim=1)
            test_acc += (test_preds_class == y).sum().item()/len(y_pred_logits)
    
        test_loss += test_loss/len(dataloader)
        test_acc += test_acc/len(dataloader)
        return test_acc, test_loss


def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device) -> Dict[str, float]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for 
    each epoch.
    In the form: {train_loss: [...],
                  train_acc: [...],
                  test_loss: [...],
                  test_acc: [...]} 
    For example if training for epochs=2: 
                 {train_loss: [2.0616, 1.0537],
                  train_acc: [0.3945, 0.3945],
                  test_loss: [1.2641, 1.5706],
                  test_acc: [0.3400, 0.2973]} 
    """
    results = {"train_loss":[], "train_acc":[], "test_loss":[], "test_acc":[]}

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model, 
                                          dataloader=train_dataloader, 
                                          optimizer=optimizer, 
                                          loss_fn=loss_fn, 
                                          device=device)

        test_loss, train_acc = train_step(model=model, 
                                          dataloader=train_dataloader, 
                                          optimizer=optimizer, 
                                          loss_fn=loss_fn, 
                                          device=device)
        
        print(f"Epoch: {epoch+1} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    # Return the filled results at the end of the epochs
    return results  
    

Overwriting going_modular/engine.py


In [27]:
# Import engine.py
from going_modular import engine

# Use train() by calling it from engine.py
engine.train(...)

#### 4. Creating a function to save the model utils.py

In [ ]:
,,.